In [76]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [77]:
!pip install ultralytics

In [78]:
!nvidia-smi

Fri May 29 20:06:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [79]:

!ls /content/dataset_yolo_fixed
!ls /content/dataset_yolo_fixed/images/train | head
!ls /content/dataset_yolo_fixed/labels/train | head

dataset.yaml  images  labels
carlong_0001.png
carlong_0002.png
carlong_0003.png
carlong_0005.png
carlong_0006.png
carlong_0007.png
carlong_0008.png
carlong_0009.png
carlong_0011.png
carlong_0012.png
carlong_0001.txt
carlong_0002.txt
carlong_0003.txt
carlong_0005.txt
carlong_0006.txt
carlong_0007.txt
carlong_0008.txt
carlong_0009.txt
carlong_0011.txt
carlong_0012.txt


In [80]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/dataset_yolo_fixed/dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project="/content/drive/MyDrive/yolo_plate_train_fixed",
    name="plate_model_fixed",
    save=True,
    save_period=5,
    device=0
)

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_yolo_fixed/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=plate_model_fixed-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_m

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x787f5e4691c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [84]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/yolo_plate_train_fixed/plate_model_fixed-3/weights/best.pt")

metrics = model.val(
    data="/content/dataset_yolo_fixed/dataset.yaml",
    split="test",
    imgsz=640,
    conf=0.25,
    iou=0.7,
    plots=True,
    project="/content/drive/MyDrive/yolo_plate_train_fixed",
    name="test_fixed"
)

print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 46.9±25.8 MB/s, size: 134.8 KB)
val: Scanning /content/dataset_yolo_fixed/labels/test... 473 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 473/473 564.3it/s 0.8s
val: New cache created: /content/dataset_yolo_fixed/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 30/30 3.3it/s 9.1s
                   all        473        553       0.98      0.989      0.994      0.907
Speed: 2.4ms preprocess, 4.1ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to /content/drive/MyDrive/yolo_plate_train_fixed/test_fixed
Precision: 0.980275850500873
Recall: 0.9891500904159132
mAP50: 0.9943848975366313
mAP50-95: 0.9071009057662284
